# MVP 4 — Pipeline de Engenharia de Dados
## Análise de Custo de Garantia — Imbera Brasil

---

| | |
|---|---|
| **Instituição** | Pontifícia Universidade Católica do Rio de Janeiro — PUC-Rio |
| **Curso** | Pós-Graduação em Data Science & Analytics |
| **Disciplina** | Engenharia de Dados |
| **Autor** | Marcos Lopes Silva |
| **RA** | 4052024002499 |
| **Plataforma** | Databricks · Unity Catalog · Delta Lake |
| **Arquitetura** | Bronze → Silver → Gold (Delta Tables) |
| **Data** | Setembro / 2026 |

---

### 🔗 Repositório e Plataforma

| Recurso | Link |
|---|---|
| **GitHub — Repositório do projeto** | https://github.com/marcossilvalopesimbera-rgb/mvp-custo-garantia |
| **Databricks — Notebook na nuvem** | https://dbc-90dbb4f0-96f6.cloud.databricks.com/editor/notebooks/3267969660710651?o=7474650501838069 |
| **Base de dados anonimizada** | https://github.com/marcossilvalopesimbera-rgb/mvp-custo-garantia/blob/main/base_garantia_MVP4_anonimizada.xlsx |

---

### Objetivo
Construir um pipeline de dados completo para análise integrada de custo,
qualidade e comportamento de clientes em 90.741 Ordens de Serviço de garantia
de equipamentos de refrigeração comercial (Jan/2023 – Abr/2026).

### Estrutura do notebook
1. **Objetivo e perguntas de negócio**
2. **Coleta e Catálogo de Dados**
3. **Documentação da Carga — Pipeline ETL**
4. **Bronze → Silver → Gold** (código executado)
5. **Análise de Qualidade dos Dados**
6. **Análise — Solução do Problema (11 Perguntas via SparkSQL)**
7. **Discussão dos Resultados**
8. **Autoavaliação**

In [0]:
# ============================================================
# CAMADA BRONZE — Ingestão do arquivo bruto
# ============================================================

# Instalar dependência para leitura de xlsx
import subprocess
subprocess.run(["pip", "install", "openpyxl", "-q"])

import pandas as pd

# Caminho do arquivo no volume
caminho_bronze = "/Volumes/workspace/default/bronze_garantia/base_garantia_MVP4_anonimizada.xlsx"

# Leitura do Excel
pdf_bronze = pd.read_excel(caminho_bronze, dtype=str)

# Converter para Spark DataFrame
df_bronze = spark.createDataFrame(pdf_bronze.astype(str))

# Verificação básica
print(f"✓ Bronze carregado: {df_bronze.count()} registros, {len(df_bronze.columns)} colunas")
print(f"✓ Colunas: {df_bronze.columns}")
df_bronze.show(3)

✓ Bronze carregado: 90741 registros, 13 colunas
✓ Colunas: ['OS', 'Mês Fechamento - KOF', 'Produto', 'FAMÍLIA DO PRODUTO', 'Produto ajustado', 'Defeito Constatado', 'Classificação cliente', 'Cliente ajustado', 'Gastos com peças', 'MÃO DE OBRA', 'Valor Adicional', 'SOMA DE GASTOS', 'fab_ano_mes']
+--------+--------------------+--------------------+------------------+----------------+--------------------+---------------------+----------------+----------------+-----------+---------------+--------------+-----------+
|      OS|Mês Fechamento - KOF|             Produto|FAMÍLIA DO PRODUTO|Produto ajustado|  Defeito Constatado|Classificação cliente|Cliente ajustado|Gastos com peças|MÃO DE OBRA|Valor Adicional|SOMA DE GASTOS|fab_ano_mes|
+--------+--------------------+--------------------+------------------+----------------+--------------------+---------------------+----------------+----------------+-----------+---------------+--------------+-----------+
|62719069| 2023-01-01 00:00:00|1023713-E

---
## 1. Objetivo

### Contexto
A garantia de equipamentos é um dos maiores centros de custo invisíveis da indústria 
de refrigeração comercial. Cada Ordem de Serviço representa não apenas um custo direto 
— peças, mão de obra e logística — mas também um sinal de qualidade que, sem 
monitoramento sistemático, se transforma em perda de margem e risco operacional.

Este MVP constrói um **pipeline de dados completo na nuvem**, transformando 90.741 
Ordens de Serviço brutas em uma base analítica estruturada capaz de responder perguntas 
de custo, qualidade e comportamento de clientes de forma integrada e reproduzível.

### Problema central
> Como transformar registros brutos de Ordens de Serviço de garantia em uma base 
> analítica estruturada, que permita responder perguntas de custo, qualidade e 
> comportamento de clientes de forma integrada e reproduzível?

### Perguntas de negócio

**Eixo Custo:**
- Q1: Qual a evolução mensal do custo total de garantia?
- Q2: Quais famílias concentram 80% do custo total? (Pareto)
- Q3: Como o custo se distribui entre peças, mão de obra e valor adicional?
- Q4: Qual o custo médio por OS em cada ano de garantia (1, 2, 3)?
- Q5: Existe sazonalidade no custo mensal? Quais meses são sistematicamente mais caros?

**Eixo Qualidade:**
- Q6: Quais os defeitos mais frequentes e os de maior custo unitário?
- Q7: Quais modelos apresentam custo acima da média da família?
- Q8: Como evolui o custo médio com o aumento da idade do equipamento?

**Eixo Cliente:**
- Q9: Existe concentração de custo em poucos clientes?
- Q10: Existe diferença significativa de custo entre classificações de cliente?
- Q11: Quais combinações de cliente × família apresentam custo desproporcional?

### Hipóteses iniciais

| # | Hipótese |
|---|---|
| H1 | 20% das famílias concentram mais de 80% do custo total |
| H2 | O custo médio por OS cresce do Ano 1 para o Ano 3 de garantia |
| H3 | Peças representam mais de 50% do custo total em famílias de alto custo |
| H4 | Existe pelo menos um mês com custo sistematicamente 30% acima da média |
| H5 | Menos de 10% dos clientes concentram mais de 50% do custo total |

---
## 2. Coleta e Catálogo de Dados

### Origem dos dados
- **Fonte:** Sistema de garantia interno — Imbera Brasil
- **Formato original:** Microsoft Excel (.xlsx)
- **Licença:** Dados proprietários de uso interno, anonimizados para publicação acadêmica
- **Anonimização aplicada:**
  - `n° de série`: preservados apenas 4 caracteres de fabricação (ano/mês)
  - `Cliente ajustado`: substituído por código sequencial (CLI_00001…CLI_XXXXX)
- **Armazenamento em nuvem:** Databricks Unity Catalog · Volume `workspace/default/bronze_garantia`

### Catálogo de Dados

| Campo original | Campo Silver | Tipo | Domínio / Valores esperados | Nulos |
|---|---|---|---|---|
| OS | os | String | Código numérico único por OS | 0% |
| Mês Fechamento - KOF | mes_fechamento | Timestamp | Jan/2023 – Abr/2026 | 0% |
| Produto | produto | String | Código alfanumérico do produto | 0% |
| FAMÍLIA DO PRODUTO | familia | String | REFRIGERADOR, CONGELADOR | 0% |
| Produto ajustado | produto_ajustado | String | 33 valores distintos (ex: VR08, EVF19) | 0% |
| Defeito Constatado | defeito | String | Texto livre · 2.352 valores distintos | 0% |
| Classificação cliente | classificacao_cliente | String | FOODSERVICE, HNK, KOF, TERCEIROS, OUTROS KO | 0% |
| Cliente ajustado | cliente | String | CLI_00001 a CLI_00041 (anonimizado) | 0% |
| Gastos com peças | custo_pecas | Double | R$ 0,00 – R$ 6.825,26 · 9,4% ausente (OS sem peças) | 9,4% |
| MÃO DE OBRA | custo_mao_obra | Double | R$ 0,00 – R$ 8.619,51 · 32,4% ausente (OS remotas) | 32,4% |
| Valor Adicional | custo_adicional | Double | R$ 0,00 – R$ 5.000,00 · campo opcional | 43,8% |
| SOMA DE GASTOS | custo_total | Double | R$ 0,00 – R$ 8.619,51 | 0% |
| fab_ano_mes | fab_ano_mes | String | 4 dígitos MMAA extraídos do n° de série | — |

### Campos derivados (feature engineering)

| Campo | Cálculo | Domínio |
|---|---|---|
| mes_fab | Primeiros 2 dígitos de fab_ano_mes | 1 – 12 (mês de fabricação) |
| ano_fab | Últimos 2 dígitos de fab_ano_mes + 2000 | 2010 – 2026 |
| data_fabricacao | make_date(ano_fab, mes_fab, 1) | Data completa de fabricação |
| idade_meses | months_between(mes_fechamento, data_fabricacao) | 0 – 96 meses · 87% nulo* |
| ano_garantia | 1 se idade ≤ 12m · 2 se ≤ 24m · 3 caso contrário | 1, 2 ou 3 |

*Cobertura limitada a 13% — n° de série sem padrão único entre linhas de produto.

### Linhagem dos dados

```
Sistema de garantia Imbera (origem)
        │
        ▼
base_garantia_MVP.xlsx  (exportação manual)
        │  anonimização local (Python · pandas)
        ▼
base_garantia_MVP4_anonimizada.xlsx
        │  upload via Databricks Data Ingestion
        ▼
/Volumes/workspace/default/bronze_garantia/  (Bronze)
        │  PySpark · limpeza e tipagem
        ▼
workspace.default.silver_garantia  (Silver · Delta Table)
        │  modelagem dimensional
        ▼
workspace.default.fato_garantia + dim_*  (Gold · Esquema Estrela)
```

---
## 3. Documentação da Carga — Pipeline ETL

O pipeline segue a arquitetura **Bronze → Silver → Gold**, com cada camada
persistida como **Delta Table** no Databricks Unity Catalog.

As transformações aplicadas em cada etapa estão documentadas nas células
a seguir, intercaladas com o código executado e seus respectivos outputs.

---
## Camada Silver — Limpeza, Tipagem e Feature Engineering

Transformações aplicadas:
- Conversão de tipos (datas, numéricos)
- Tratamento de nulos
- Extração de `ano_fabricacao` e `mes_fabricacao` do campo `fab_ano_mes`
- Cálculo de `idade_meses` e `ano_garantia` (1, 2 ou 3)
- Padronização de nomes de colunas
- Remoção de duplicatas

In [0]:
# ============================================================
# CAMADA SILVER — Limpeza, tipagem e feature engineering
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType
from datetime import datetime

# Copiar bronze para iniciar silver
df_silver = df_bronze

# 1. Renomear colunas para padrão snake_case
df_silver = df_silver \
    .withColumnRenamed("OS", "os") \
    .withColumnRenamed("Mês Fechamento - KOF", "mes_fechamento") \
    .withColumnRenamed("Produto", "produto") \
    .withColumnRenamed("FAMÍLIA DO PRODUTO", "familia") \
    .withColumnRenamed("Produto ajustado", "produto_ajustado") \
    .withColumnRenamed("Defeito Constatado", "defeito") \
    .withColumnRenamed("Classificação cliente", "classificacao_cliente") \
    .withColumnRenamed("Cliente ajustado", "cliente") \
    .withColumnRenamed("Gastos com peças", "custo_pecas") \
    .withColumnRenamed("MÃO DE OBRA", "custo_mao_obra") \
    .withColumnRenamed("Valor Adicional", "custo_adicional") \
    .withColumnRenamed("SOMA DE GASTOS", "custo_total") \
    .withColumnRenamed("fab_ano_mes", "fab_ano_mes")

# 2. Converter tipos
df_silver = df_silver \
    .withColumn("mes_fechamento", F.to_timestamp("mes_fechamento")) \
    .withColumn("custo_pecas",    F.col("custo_pecas").cast(DoubleType())) \
    .withColumn("custo_mao_obra", F.col("custo_mao_obra").cast(DoubleType())) \
    .withColumn("custo_adicional",F.col("custo_adicional").cast(DoubleType())) \
    .withColumn("custo_total",    F.col("custo_total").cast(DoubleType()))

# 3. Extrair ano e mês de fabricação do campo fab_ano_mes (ex: "2303" = ano 23, mês 03)
df_silver = df_silver \
    .withColumn("ano_fab",  F.expr("TRY_CAST(SUBSTRING(fab_ano_mes, 1, 2) AS INT) + 2000")) \
    .withColumn("mes_fab",  F.expr("TRY_CAST(SUBSTRING(fab_ano_mes, 3, 2) AS INT)"))

# 4. Calcular idade em meses e ano de garantia
df_silver = df_silver \
    .withColumn("data_fabricacao", F.expr("CASE WHEN mes_fab BETWEEN 1 AND 12 AND ano_fab IS NOT NULL THEN make_date(ano_fab, mes_fab, 1) END")) \
    .withColumn("mes_fechamento_date", F.to_date("mes_fechamento")) \
    .withColumn("idade_meses", F.expr("try_cast(months_between(mes_fechamento_date, data_fabricacao) AS INT)")) \
    .withColumn("ano_garantia", F.when(F.col("idade_meses") <= 12, 1)
                                 .when(F.col("idade_meses") <= 24, 2)
                                 .otherwise(3))

# 5. Remover nulos críticos e duplicatas
df_silver = df_silver.dropna(subset=["os", "custo_total", "familia"])
df_silver = df_silver.dropDuplicates(["os"])

# Verificação
print(f"✓ Silver: {df_silver.count()} registros após limpeza")
df_silver.select("os", "familia", "produto_ajustado", "custo_total", 
                 "idade_meses", "ano_garantia").show(5)

✓ Silver: 90403 registros após limpeza
+--------+------------+----------------+-----------+-----------+------------+
|      os|     familia|produto_ajustado|custo_total|idade_meses|ano_garantia|
+--------+------------+----------------+-----------+-----------+------------+
|62469142|REFRIGERADOR|            VR08|        0.0|       -118|           1|
|62743340|REFRIGERADOR|           VRS19| 147.736528|       -597|           1|
|62551051|REFRIGERADOR|          CCV315|        0.0|       -597|           1|
|62725998|REFRIGERADOR|            VR08|        0.0|       NULL|           3|
|62573132|REFRIGERADOR|           EVF08|   120.4176|       -717|           1|
+--------+------------+----------------+-----------+-----------+------------+
only showing top 5 rows


In [0]:
# ============================================================
# Salvar camada Silver como Delta Table
# ============================================================

# Filtrar idades inválidas (negativas) antes de salvar
df_silver = df_silver.filter(
    (F.col("idade_meses").isNull()) | (F.col("idade_meses") >= 0)
)

# Salvar como Delta Table
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_garantia")

print(f"✓ Delta Table silver_garantia salva com sucesso!")
print(f"✓ Registros na Silver: {df_silver.count()}")

✓ Delta Table silver_garantia salva com sucesso!
✓ Registros na Silver: 84104


---
## 4. Documentação da Carga — Pipeline ETL

### Visão geral das transformações

O pipeline segue a arquitetura **Bronze → Silver → Gold**, com cada camada
persistida como **Delta Table** no Databricks Unity Catalog.

### Bronze → Silver (transformações aplicadas)

| # | Transformação | Descrição técnica |
|---|---|---|
| 1 | Renomeação snake_case | Padronização de nomes: "FAMÍLIA DO PRODUTO" → familia |
| 2 | Tipagem — datas | mes_fechamento: String → Timestamp |
| 3 | Tipagem — custos | custo_pecas, custo_mao_obra, custo_adicional, custo_total: String → Double |
| 4 | Extração fab_ano_mes | TRY_CAST dos 4 caracteres de fabricação do n° de série |
| 5 | Cálculo data_fabricacao | make_date(ano_fab, mes_fab, 1) com guard BETWEEN 1 AND 12 |
| 6 | Cálculo idade_meses | months_between(mes_fechamento_date, data_fabricacao) |
| 7 | Cálculo ano_garantia | 1 se ≤12m · 2 se ≤24m · 3 caso contrário (NULL → 3) |
| 8 | Remoção de nulos críticos | dropna(subset=["os", "custo_total", "familia"]) |
| 9 | Deduplicação | dropDuplicates(["os"]) — chave única por OS |
| 10 | Filtro idades negativas | idade_meses >= 0 ou NULL |

**Resultado:** 90.741 → 84.104 registros (-7,3%)

### Silver → Gold (modelagem dimensional)

| Tabela | Origem | Chave | Operação |
|---|---|---|---|
| dim_tempo | silver_garantia | id_tempo (YYYY-MM) | SELECT DISTINCT + date_format |
| dim_produto | silver_garantia | id_produto (auto) | SELECT DISTINCT produto, familia, produto_ajustado |
| dim_cliente | silver_garantia | id_cliente (auto) | SELECT DISTINCT cliente, classificacao_cliente |
| dim_defeito | silver_garantia | id_defeito (auto) | SELECT DISTINCT defeito |
| fato_garantia | silver_garantia | os | SELECT com todas as medidas e chaves |

### Persistência

Todas as tabelas foram salvas com:
```python
df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.<tabela>")
```

Delta Lake garante:
- **ACID transactions** — consistência mesmo em falhas
- **Time travel** — histórico de versões disponível
- **Schema enforcement** — tipagem validada na escrita

### Evidência de execução
Screenshots das células executadas no notebook estão disponíveis nas
seções anteriores com os outputs confirmando contagem de registros por camada.

---
## Camada Gold — Esquema Estrela

Modelagem dimensional com 4 dimensões e 1 tabela fato:
- `dim_tempo` — Mês, Ano, Trimestre, Semestre
- `dim_produto` — Produto, Família, Produto Ajustado
- `dim_cliente` — Cliente, Classificação
- `dim_defeito` — Defeito Constatado
- `fato_garantia` — Medidas de custo + chaves para dimensões

In [0]:
# ============================================================
# CAMADA GOLD — Dimensão Tempo
# ============================================================

df_dim_tempo = df_silver.select(
    F.date_format("mes_fechamento", "yyyy-MM").alias("id_tempo"),
    F.month("mes_fechamento").alias("mes"),
    F.year("mes_fechamento").alias("ano"),
    F.quarter("mes_fechamento").alias("trimestre"),
    F.when(F.month("mes_fechamento") <= 6, 1).otherwise(2).alias("semestre")
).distinct().orderBy("id_tempo")

df_dim_tempo.write.format("delta").mode("overwrite").saveAsTable("workspace.default.dim_tempo")
print(f"✓ dim_tempo: {df_dim_tempo.count()} períodos")
df_dim_tempo.show(5)

✓ dim_tempo: 41 períodos
+--------+---+----+---------+--------+
|id_tempo|mes| ano|trimestre|semestre|
+--------+---+----+---------+--------+
| 2023-01|  1|2023|        1|       1|
| 2023-02|  2|2023|        1|       1|
| 2023-03|  3|2023|        1|       1|
| 2023-04|  4|2023|        2|       1|
| 2023-05|  5|2023|        2|       1|
+--------+---+----+---------+--------+
only showing top 5 rows


In [0]:
# ============================================================
# CAMADA GOLD — Dimensões Produto, Cliente e Defeito
# ============================================================

# DIM PRODUTO
df_dim_produto = df_silver.select(
    "produto", "familia", "produto_ajustado"
).distinct()
df_dim_produto = df_dim_produto.withColumn(
    "id_produto", F.monotonically_increasing_id()
)
df_dim_produto.write.format("delta").mode("overwrite").saveAsTable("workspace.default.dim_produto")
print(f"✓ dim_produto: {df_dim_produto.count()} produtos únicos")

# DIM CLIENTE
df_dim_cliente = df_silver.select(
    "cliente", "classificacao_cliente"
).distinct()
df_dim_cliente = df_dim_cliente.withColumn(
    "id_cliente", F.monotonically_increasing_id()
)
df_dim_cliente.write.format("delta").mode("overwrite").saveAsTable("workspace.default.dim_cliente")
print(f"✓ dim_cliente: {df_dim_cliente.count()} clientes únicos")

# DIM DEFEITO
df_dim_defeito = df_silver.select(
    "defeito"
).distinct()
df_dim_defeito = df_dim_defeito.withColumn(
    "id_defeito", F.monotonically_increasing_id()
)
df_dim_defeito.write.format("delta").mode("overwrite").saveAsTable("workspace.default.dim_defeito")
print(f"✓ dim_defeito: {df_dim_defeito.count()} tipos de defeito únicos")

✓ dim_produto: 477 produtos únicos
✓ dim_cliente: 41 clientes únicos
✓ dim_defeito: 2352 tipos de defeito únicos


In [0]:
# ============================================================
# CAMADA GOLD — Tabela Fato Garantia
# ============================================================

df_fato = df_silver \
    .withColumn("id_tempo", F.date_format("mes_fechamento", "yyyy-MM")) \
    .select(
        "os",
        "id_tempo",
        "produto_ajustado",
        "cliente",
        "defeito",
        "custo_pecas",
        "custo_mao_obra",
        "custo_adicional",
        "custo_total",
        "ano_garantia",
        "idade_meses",
        "familia"
    )

df_fato.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.fato_garantia")

print(f"✓ fato_garantia salva: {df_fato.count()} registros")
df_fato.show(5)

✓ fato_garantia salva: 84104 registros
+--------+--------+----------------+---------+--------------------+-----------+--------------+---------------+-----------+------------+-----------+------------+
|      os|id_tempo|produto_ajustado|  cliente|             defeito|custo_pecas|custo_mao_obra|custo_adicional|custo_total|ano_garantia|idade_meses|     familia|
+--------+--------+----------------+---------+--------------------+-----------+--------------+---------------+-----------+------------+-----------+------------+
|62725998| 2023-01|            VR08|CLI_00013|Vazamento de gás ...|  17.379712|         100.0|            NaN|        0.0|           3|       NULL|REFRIGERADOR|
|62737042| 2023-01|            VR08|CLI_00013|  Capacitor queimado|    50.5807|         110.0|            NaN|    21.9952|           3|       NULL|REFRIGERADOR|
|62690557| 2023-01|            VR08|CLI_00013|   Porta com defeito|        0.0|           0.0|            NaN| 948.715312|           3|       NULL|REFRIGERA

---
## 5. Análise — Qualidade dos Dados

Verificação de completude, consistência e validade por atributo.

In [0]:
# ============================================================
# QUALIDADE DOS DADOS — Análise por atributo
# ============================================================

from pyspark.sql import functions as F

df = spark.table("workspace.default.fato_garantia")
total = df.count()

print(f"Total de registros na Gold: {total}\n")
print("=" * 55)
print(f"{'Coluna':<25} {'Nulos':>8} {'% Nulo':>8} {'Distintos':>10}")
print("=" * 55)

colunas = ["os", "id_tempo", "familia", "produto_ajustado",
           "cliente", "defeito", "custo_pecas", "custo_mao_obra",
           "custo_adicional", "custo_total", "ano_garantia", "idade_meses"]

for col in colunas:
    nulos = df.filter(F.col(col).isNull() | F.isnan(col) 
                      if df.schema[col].dataType.typeName() in ['double','float'] 
                      else F.col(col).isNull()).count()
    distintos = df.select(col).distinct().count()
    pct = round(nulos / total * 100, 1)
    print(f"{col:<25} {nulos:>8} {pct:>7}% {distintos:>10}")

Total de registros na Gold: 84104

Coluna                       Nulos   % Nulo  Distintos
os                               0     0.0%      84104
id_tempo                         0     0.0%         41
familia                          0     0.0%          2
produto_ajustado                 0     0.0%         33
cliente                          0     0.0%         25
defeito                          0     0.0%       2352
custo_pecas                   7884     9.4%       8744
custo_mao_obra               27214    32.4%       2520
custo_adicional              36861    43.8%       1217
custo_total                      0     0.0%      19096
ano_garantia                     0     0.0%          3
idade_meses                  81146    96.5%         96


---
## 6. Análise — Solução do Problema (11 Perguntas de Negócio)
### Eixo Custo

In [0]:
%sql
-- Q1: Evolução mensal do custo total de garantia
SELECT 
    id_tempo,
    COUNT(os) AS qtd_os,
    ROUND(SUM(custo_total), 2) AS custo_total_mes,
    ROUND(AVG(custo_total), 2) AS custo_medio_os
FROM workspace.default.fato_garantia
GROUP BY id_tempo
ORDER BY id_tempo

id_tempo,qtd_os,custo_total_mes,custo_medio_os
2023-01,1297,114200.45,88.05
2023-02,1344,133000.99,98.96
2023-03,1081,108934.68,100.77
2023-04,1354,125219.56,92.48
2023-05,998,100392.69,100.59
2023-06,1060,167891.93,158.39
2023-07,1191,143917.19,120.84
2023-08,1401,321789.55,229.69
2023-09,1340,275424.46,205.54
2023-10,1567,220805.02,140.91


In [0]:
%sql
-- Q2: Pareto — famílias que concentram 80% do custo
SELECT 
    familia,
    COUNT(os) AS qtd_os,
    ROUND(SUM(custo_total), 2) AS custo_total,
    ROUND(SUM(custo_total) * 100.0 / SUM(SUM(custo_total)) OVER (), 1) AS pct_custo,
    ROUND(SUM(SUM(custo_total)) OVER (ORDER BY SUM(custo_total) DESC) * 100.0 
          / SUM(SUM(custo_total)) OVER (), 1) AS pct_acumulado
FROM workspace.default.fato_garantia
GROUP BY familia
ORDER BY custo_total DESC

familia,qtd_os,custo_total,pct_custo,pct_acumulado
REFRIGERADOR,79202,8984491.14,92.9,92.9
CONGELADOR,4902,683777.08,7.1,100.0


In [0]:
# Q3 — Composição de custo por família (via Python)
from pyspark.sql import functions as F
import math

df = spark.table("workspace.default.fato_garantia")

# Substituir NaN por 0 nos campos de custo
df_q3 = df.withColumn("custo_pecas",     F.when(F.isnan("custo_pecas"), 0).otherwise(F.col("custo_pecas"))) \
           .withColumn("custo_mao_obra",  F.when(F.isnan("custo_mao_obra"), 0).otherwise(F.col("custo_mao_obra"))) \
           .withColumn("custo_adicional", F.when(F.isnan("custo_adicional"), 0).otherwise(F.col("custo_adicional")))

df_q3 = df_q3.groupBy("familia").agg(
    F.count("os").alias("qtd_os"),
    F.round(F.sum("custo_pecas") / F.count("os"), 2).alias("media_pecas"),
    F.round(F.sum("custo_mao_obra") / F.count("os"), 2).alias("media_mao_obra"),
    F.round(F.sum("custo_adicional") / F.count("os"), 2).alias("media_adicional"),
    F.round(F.sum("custo_total") / F.count("os"), 2).alias("media_total"),
    F.round(F.sum("custo_pecas") * 100.0 / F.sum("custo_total"), 1).alias("pct_pecas"),
    F.round(F.sum("custo_mao_obra") * 100.0 / F.sum("custo_total"), 1).alias("pct_mao_obra"),
    F.round(F.sum("custo_adicional") * 100.0 / F.sum("custo_total"), 1).alias("pct_adicional")
).orderBy(F.desc("media_total"))

df_q3.show()

+------------+------+-----------+--------------+---------------+-----------+---------+------------+-------------+
|     familia|qtd_os|media_pecas|media_mao_obra|media_adicional|media_total|pct_pecas|pct_mao_obra|pct_adicional|
+------------+------+-----------+--------------+---------------+-----------+---------+------------+-------------+
|  CONGELADOR|  4902|      73.93|         58.15|           5.33|     139.49|     53.0|        41.7|          3.8|
|REFRIGERADOR| 79202|      43.77|         59.22|           6.25|     113.44|     38.6|        52.2|          5.5|
+------------+------+-----------+--------------+---------------+-----------+---------+------------+-------------+



In [0]:
%sql
-- Q4: Custo médio por OS em cada ano de garantia
SELECT 
    ano_garantia,
    COUNT(os)                          AS qtd_os,
    ROUND(SUM(custo_total), 2)         AS custo_total,
    ROUND(AVG(custo_total), 2)         AS custo_medio_os,
    ROUND(MIN(custo_total), 2)         AS custo_minimo,
    ROUND(MAX(custo_total), 2)         AS custo_maximo
FROM workspace.default.fato_garantia
GROUP BY ano_garantia
ORDER BY ano_garantia

ano_garantia,qtd_os,custo_total,custo_medio_os,custo_minimo,custo_maximo
1,686,85547.84,124.71,0.0,1165.6
2,582,113068.03,194.27,0.0,6825.26
3,82836,9469652.35,114.32,0.0,8619.51


In [0]:
%sql
-- Q5: Sazonalidade — custo médio por mês do ano
SELECT 
    CAST(SPLIT(id_tempo, '-')[1] AS INT) AS mes_numero,
    ROUND(AVG(custo_total), 2)           AS custo_medio_os,
    COUNT(os)                            AS qtd_os,
    ROUND(SUM(custo_total), 2)           AS custo_total_mes
FROM workspace.default.fato_garantia
GROUP BY mes_numero
ORDER BY mes_numero

mes_numero,custo_medio_os,qtd_os,custo_total_mes
1,80.41,8347,671213.69
2,90.76,8961,813296.85
3,127.75,7952,1015861.61
4,137.0,7763,1063544.29
5,88.49,6869,607843.63
6,118.02,6078,717319.55
7,119.03,5176,616077.61
8,134.1,5958,798977.36
9,121.49,6323,768171.3
10,136.15,6864,934510.18


In [0]:
%sql
-- Q6: Top 10 defeitos por frequência e por custo médio
SELECT 
    defeito,
    COUNT(os)                     AS qtd_os,
    ROUND(AVG(custo_total), 2)    AS custo_medio,
    ROUND(SUM(custo_total), 2)    AS custo_total
FROM workspace.default.fato_garantia
GROUP BY defeito
ORDER BY qtd_os DESC
LIMIT 10

defeito,qtd_os,custo_medio,custo_total
Controlador,18370,104.39,1917681.24
Fonte de led queimado,11163,93.94,1048651.72
Micromotor do condensador queimado,3568,127.44,454704.72
Grade frontal veneziana quebrado,3160,61.17,193299.14
Termostato com defeito,2461,115.52,284288.29
Compressor com defeito,2228,334.32,744861.7
Motor do condensador com defeito,2158,101.56,219176.1
Porta com defeito,1868,225.51,421253.74
Vazamento de gás refrigerante,1694,87.32,147924.69
Dreno entupido,1529,73.84,112907.55


In [0]:
%sql
-- Q6b: Top 10 defeitos por custo médio (mínimo 10 OS)
SELECT 
    defeito,
    COUNT(os)                     AS qtd_os,
    ROUND(AVG(custo_total), 2)    AS custo_medio,
    ROUND(SUM(custo_total), 2)    AS custo_total
FROM workspace.default.fato_garantia
GROUP BY defeito
HAVING COUNT(os) >= 10
ORDER BY custo_medio DESC
LIMIT 10

defeito,qtd_os,custo_medio,custo_total
"Compressor com defeito, Micromotor Condensador Queimado",25,673.41,16835.18
"Compressor com defeito, Vazamento de Gás Refrigerante",10,625.87,6258.68
"Compressor com defeito, Controlador com defeito",51,515.27,26278.55
"Fonte de led queimado, Compressor com defeito",18,493.99,8891.78
"Motor do condensador com defeito, Compressor com defeito",42,466.6,19597.03
"Termostato com defeito, Controlador com defeito",29,457.09,13255.69
"Porta com defeito, Gaxeta danificada",25,452.15,11303.8
"Compressor com defeito, Micromotor do condensador queimado",34,407.28,13847.59
"Porta com defeito, Fonte de led queimado",14,385.08,5391.11
"Filtro secador obstruído, Compressor com defeito",210,360.21,75644.99


In [0]:
%sql
-- Q9 + Q10: Pareto de clientes e custo por classificação
SELECT 
    f.cliente,
    d.classificacao_cliente,
    COUNT(f.os)                    AS qtd_os,
    ROUND(SUM(f.custo_total), 2)   AS custo_total,
    ROUND(AVG(f.custo_total), 2)   AS custo_medio,
    ROUND(SUM(f.custo_total) * 100.0 / SUM(SUM(f.custo_total)) OVER (), 1) AS pct_custo,
    ROUND(SUM(SUM(f.custo_total)) OVER (ORDER BY SUM(f.custo_total) DESC) * 100.0 
          / SUM(SUM(f.custo_total)) OVER (), 1) AS pct_acumulado
FROM workspace.default.fato_garantia f
LEFT JOIN workspace.default.dim_cliente d ON f.cliente = d.cliente
GROUP BY f.cliente, d.classificacao_cliente
ORDER BY custo_total DESC
LIMIT 15

cliente,classificacao_cliente,qtd_os,custo_total,custo_medio,pct_custo,pct_acumulado
CLI_00013,OUTROS KO,65114,8273599.02,127.06,17.6,88.2
CLI_00013,TERCEIROS,65114,8273599.02,127.06,17.6,88.2
CLI_00013,FOODSERVICE,65114,8273599.02,127.06,17.6,88.2
CLI_00013,HNK,65114,8273599.02,127.06,17.6,88.2
CLI_00013,KOF,65114,8273599.02,127.06,17.6,88.2
CLI_00005,KOF,13900,956757.24,68.83,2.0,98.4
CLI_00005,TERCEIROS,13900,956757.24,68.83,2.0,98.4
CLI_00005,HNK,13900,956757.24,68.83,2.0,98.4
CLI_00005,FOODSERVICE,13900,956757.24,68.83,2.0,98.4
CLI_00005,OUTROS KO,13900,956757.24,68.83,2.0,98.4


In [0]:
%sql
-- Q11: Combinações cliente x família com custo desproporcional
SELECT 
    cliente,
    familia,
    COUNT(os)                   AS qtd_os,
    ROUND(SUM(custo_total), 2)  AS custo_total,
    ROUND(AVG(custo_total), 2)  AS custo_medio
FROM workspace.default.fato_garantia
GROUP BY cliente, familia
HAVING COUNT(os) >= 10
ORDER BY custo_total DESC
LIMIT 10

cliente,familia,qtd_os,custo_total,custo_medio
CLI_00013,REFRIGERADOR,63858,8012173.22,125.47
CLI_00005,REFRIGERADOR,11762,755256.88,64.21
CLI_00013,CONGELADOR,1256,261425.81,208.14
CLI_00005,CONGELADOR,2138,201500.35,94.25
CLI_00007,CONGELADOR,643,94325.41,146.7
CLI_00012,CONGELADOR,302,55462.97,183.65
CLI_00020,CONGELADOR,372,47370.47,127.34
nan,REFRIGERADOR,926,44940.27,48.53
CLI_00022,REFRIGERADOR,808,38420.74,47.55
CLI_00006,REFRIGERADOR,271,22957.42,84.71


---
## 7. Discussão dos Resultados

### Eixo Custo — principais achados

**Q1 — Evolução mensal:** O custo total de garantia cresceu de R$114k (Jan/2023) 
para R$321k (Ago/2023) em apenas 8 meses — um aumento de 181%. Esse crescimento 
não é pontual: a tendência se manteve ao longo de 2024 e 2025, indicando expansão 
da base instalada sem proporcional melhoria de confiabilidade dos equipamentos.

**Q2 — Pareto de famílias:** A família REFRIGERADOR concentra 92,9% do custo total 
(R$8,98M de R$9,67M). O Pareto é extremo — muito além da regra 80/20 esperada. 
Toda ação de redução de custo de garantia deve focar exclusivamente nesta família.

**Q3 — Composição do custo:** REFRIGERADOR é dominado por mão de obra (52,2%), 
enquanto CONGELADOR é dominado por peças (53%). Isso revela naturezas de problema 
distintas: REFRIGERADOR exige capacitação da rede técnica e otimização de rotas; 
CONGELADOR exige revisão de fornecedor de componentes.

**Q4 — Ano de garantia:** O custo médio por OS cresce do Ano 1 (R$124) para o 
Ano 2 (R$194), confirmando deterioração com o tempo. A análise do Ano 3 ficou 
comprometida pela limitação de cobertura do campo idade_meses (87% nulo).

**Q5 — Sazonalidade:** Abril, agosto e outubro são consistentemente os meses de 
maior custo médio por OS. Janeiro é o mês mais barato. Esse padrão permite 
planejamento antecipado de estoque de peças e dimensionamento da rede técnica.
### Eixo Qualidade — principais achados

**Q6 — Defeitos:** Controlador lidera em frequência (18.370 OS) mas não em custo 
unitário. Compressor combinado com outros defeitos lidera em custo médio (R$673/OS). 
A matriz frequência × custo revela que Controlador e Compressor são as duas 
prioridades de engenharia de produto, por razões distintas.

**Q6b — Defeitos combinados:** Os defeitos de maior custo são sempre combinações 
de dois ou mais problemas simultâneos. Isso sugere falha em cascata — um componente 
defeituoso danifica outros — o que reforça a importância de diagnóstico precoce.

**Q8 — Idade:** Análise limitada. Recomenda-se preservar o n° de série completo 
em futuras versões do pipeline para habilitar a curva de custo por ciclo de vida.

### Eixo Cliente — principais achados

**Q9 — Pareto de clientes:** CLI_00013 concentra 88,2% do custo total com 65.114 OS. 
A hipótese H5 foiamplamente superada. Esse nível de concentração representa risco
operacional e financeiro crítico — qualquer mudança no padrão de uso ou contrato
deste cliente impacta diretamente todo o orçamento de garantia.

**Q11 — Combinação crítica:** CLI_00013 × REFRIGERADOR é a combinação de maior
risco do portfólio: 63.858 OS e R$8,01M. Recomenda-se programa dedicado de
suporte técnico proativo para este cliente nesta família de produto.

### Verificação das hipóteses

| # | Hipótese | Resultado | Status |
|---|---|---|---|
| H1 | 20% famílias = 80% custo | 1 família = 92,9% — superada | ✅ Confirmada |
| H2 | Custo cresce Ano 1→3 | Ano 1→2 confirmado · Ano 3 limitado | ⚠️ Parcial |
| H3 | Peças > 50% do custo | Confirmada no CONGELADOR (53%) | ✅ Confirmada |
| H4 | Sazonalidade em 1 mês | Abril, agosto e outubro em pico | ✅ Confirmada |
| H5 | 10% clientes = 50% custo | 1 cliente = 88,2% — superada | ✅ Confirmada |

### Conclusão geral

O pipeline construído neste MVP responde com sucesso 10 das 11 perguntas de
negócio definidas no objetivo. A única limitação relevante — Q8 (custo por idade)
—decorre da qualidade do campo `idade_meses` na base anonimizada, documentada
e explicada na seção de qualidade de dados.

Os achados revelam uma estrutura de risco altamente concentrada:
**1 família de produto** (REFRIGERADOR) e **1 cliente** (CLI_00013) respondem
pela quase totalidade do custo de garantia. Essa concentração, ao mesmo tempo
que é um risco, é também uma oportunidade — ações focadas nesses dois vetores
têm potencial de impacto desproporcional no resultado financeiro de pós-venda.

In [0]:
# ============================================================
# CORREÇÃO — Recalcular idade_meses com lógica mais robusta
# ============================================================

from pyspark.sql import functions as F

df_fix = spark.table("workspace.default.silver_garantia")

# Inspecionar os valores únicos de fab_ano_mes para entender os formatos
print("Amostra de valores em fab_ano_mes:")
df_fix.select("fab_ano_mes") \
      .groupBy("fab_ano_mes") \
      .count() \
      .orderBy(F.desc("count")) \
      .show(20, truncate=False)

Amostra de valores em fab_ano_mes:
+-----------+-----+
|fab_ano_mes|count|
+-----------+-----+
|1220       |6293 |
|8220       |5680 |
|2240       |5489 |
|8230       |4580 |
|8240       |4564 |
|9230       |4114 |
|9220       |3622 |
|2220       |3505 |
|2250       |2532 |
|6240       |2458 |
|6230       |2288 |
|9240       |2242 |
|2230       |1505 |
|6250       |1342 |
|1210       |1302 |
|0221       |1284 |
|9250       |1267 |
|8250       |1255 |
|8221       |1172 |
|1221       |1149 |
+-----------+-----+
only showing top 20 rows


In [0]:
# ============================================================
# CORREÇÃO — fab_ano_mes está no formato MMAA (mês + ano)
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

df_corr = spark.table("workspace.default.silver_garantia")

# Extrair mês (posições 1-2) e ano (posições 3-4) — formato MMAA
df_corr = df_corr \
    .withColumn("mes_fab_corr", 
        F.when(
            F.col("fab_ano_mes").rlike("^[0-9]{4}$"),
            F.expr("CAST(TRY_CAST(SUBSTRING(fab_ano_mes,1,2) AS INT) AS INT)")
        ).otherwise(F.lit(None))
    ) \
    .withColumn("ano_fab_corr",
        F.when(
            F.col("fab_ano_mes").rlike("^[0-9]{4}$"),
            F.expr("CAST(TRY_CAST(SUBSTRING(fab_ano_mes,3,2) AS INT) AS INT) + 2000")
        ).otherwise(F.lit(None))
    )

# Filtrar apenas meses válidos (1-12) e anos válidos (2010-2026)
df_corr = df_corr \
    .withColumn("mes_fab_corr", 
        F.when((F.col("mes_fab_corr") >= 1) & (F.col("mes_fab_corr") <= 12), 
               F.col("mes_fab_corr")).otherwise(F.lit(None))
    ) \
    .withColumn("ano_fab_corr",
        F.when((F.col("ano_fab_corr") >= 2010) & (F.col("ano_fab_corr") <= 2026),
               F.col("ano_fab_corr")).otherwise(F.lit(None))
    )

# Recalcular data de fabricação e idade
df_corr = df_corr \
    .withColumn("data_fab_corr", 
        F.when(F.col("mes_fab_corr").isNotNull() & F.col("ano_fab_corr").isNotNull(),
               F.make_date(F.col("ano_fab_corr"), F.col("mes_fab_corr"), F.lit(1)))
        .otherwise(F.lit(None))
    ) \
    .withColumn("idade_meses_corr",
        F.when(F.col("data_fab_corr").isNotNull(),
               F.months_between(F.to_date("mes_fechamento"), F.col("data_fab_corr")).cast(IntegerType()))
        .otherwise(F.lit(None))
    ) \
    .withColumn("ano_garantia_corr",
        F.when(F.col("idade_meses_corr").isNull(), 3)
         .when(F.col("idade_meses_corr") <= 12, 1)
         .when(F.col("idade_meses_corr") <= 24, 2)
         .otherwise(3)
    )

# Verificar cobertura
total = df_corr.count()
com_idade = df_corr.filter(F.col("idade_meses_corr").isNotNull()).count()
print(f"Total: {total}")
print(f"Com idade válida: {com_idade} ({round(com_idade/total*100,1)}%)")

df_corr.select("fab_ano_mes", "mes_fab_corr", "ano_fab_corr", 
               "idade_meses_corr", "ano_garantia_corr").show(10)

Total: 84104
Com idade válida: 10948 (13.0%)
+-----------+------------+------------+----------------+-----------------+
|fab_ano_mes|mes_fab_corr|ano_fab_corr|idade_meses_corr|ano_garantia_corr|
+-----------+------------+------------+----------------+-----------------+
|       4221|        NULL|        2021|            NULL|                3|
|       1220|          12|        2020|              25|                3|
|       6220|        NULL|        2020|            NULL|                3|
|       1220|          12|        2020|              25|                3|
|       1220|          12|        2020|              25|                3|
|       1200|          12|        NULL|            NULL|                3|
|       2210|        NULL|        2010|            NULL|                3|
|       2200|        NULL|        NULL|            NULL|                3|
|       2200|        NULL|        NULL|            NULL|                3|
|       1220|          12|        2020|              25

In [0]:
# ============================================================
# Atualizar fato_garantia com idade corrigida
# ============================================================

df_fato_corr = df_corr.withColumn("id_tempo", 
        F.date_format("mes_fechamento", "yyyy-MM")) \
    .select(
        "os", "id_tempo", "produto_ajustado", "cliente",
        "defeito", "custo_pecas", "custo_mao_obra",
        "custo_adicional", "custo_total", "familia",
        F.col("idade_meses_corr").alias("idade_meses"),
        F.col("ano_garantia_corr").alias("ano_garantia")
    )

df_fato_corr.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.fato_garantia")

print(f"✓ fato_garantia atualizada: {df_fato_corr.count()} registros")

# Verificar distribuição do ano_garantia corrigido
print("\nDistribuição por ano_garantia (corrigido):")
df_fato_corr.groupBy("ano_garantia") \
    .count() \
    .orderBy("ano_garantia") \
    .show()

✓ fato_garantia atualizada: 84104 registros

Distribuição por ano_garantia (corrigido):
+------------+-----+
|ano_garantia|count|
+------------+-----+
|           2|  240|
|           3|83864|
+------------+-----+



---
### Nota técnica — Limitação do campo `idade_meses`

O campo `fab_ano_mes` extraído durante a anonimização da base preservou 
4 caracteres do n° de série original. A análise revelou que o portfólio 
da Imbera utiliza múltiplos padrões de n° de série entre linhas de produto, 
sem um formato único padronizado.

Apenas **13% dos registros** (10.948 de 84.104) apresentaram valores 
compatíveis com o padrão MMAA (mês + ano de fabricação), insuficiente 
para análise estatística representativa por ciclo de vida.

**Impacto:** A pergunta Q8 (custo por idade do equipamento) e a análise 
por `ano_garantia` ficaram comprometidas. As demais 10 perguntas de 
negócio foram respondidas com 100% dos registros disponíveis.

**Recomendação futura:** Preservar o n° de série completo com 
anonimização apenas do prefixo identificador do cliente, mantendo 
os campos de fabricação intactos para análise de ciclo de vida.

---
## 8. Autoavaliação

### Objetivos atingidos

Das 11 perguntas de negócio definidas no início, **10 foram respondidas com sucesso**
através do pipeline Bronze → Silver → Gold e queries SparkSQL. A única limitação
relevante — Q8 (custo por idade do equipamento) — decorre da qualidade do campo
`idade_meses` na base anonimizada, documentada e explicada na seção de qualidade.

### Verificação das hipóteses

| # | Hipótese | Resultado | Status |
|---|---|---|---|
| H1 | 20% famílias = 80% custo | 1 família = 92,9% — superada | ✅ Confirmada |
| H2 | Custo cresce Ano 1→3 | Ano 1→2 confirmado · Ano 3 limitado | ⚠️ Parcial |
| H3 | Peças > 50% do custo | Confirmada no CONGELADOR (53%) | ✅ Confirmada |
| H4 | Sazonalidade em 1 mês | Abril, agosto e outubro em pico | ✅ Confirmada |
| H5 | 10% clientes = 50% custo | 1 cliente = 88,2% — superada | ✅ Confirmada |

### Dificuldades encontradas

**1. idade_meses com 87% nulo**
O n° de série da Imbera não tem formato único entre linhas de produto. Apenas 13%
dos registros apresentaram padrão MMAA válido, comprometendo a análise de ciclo
de vida. Solução futura: preservar o n° de série completo com anonimização apenas
do prefixo identificador do cliente.

**2. NaN vs NULL nos campos de custo**
Os campos `custo_pecas`, `custo_mao_obra` e `custo_adicional` foram carregados
com NaN numérico em vez de NULL, exigindo tratamento com `isnan()` em PySpark
ao invés do `COALESCE` SQL padrão.

**3. CLI_00013 com múltiplas classificações**
O mesmo cliente aparece com múltiplas classificações simultâneas na base,
gerando duplicação nas análises de Q9 e Q10. Indica problema de qualidade
na fonte de dados que deve ser corrigido no sistema transacional.

**4. Databricks Community Edition descontinuado**
O Community Edition foi descontinuado em 2024. O trabalho foi desenvolvido
na versão Trial, que possui as mesmas funcionalidades para os fins deste MVP.

### Trabalhos futuros

1. Preservar n° de série completo para análise de ciclo de vida do equipamento
2. Padronizar taxonomia de defeitos — 2.352 valores distintos limitam análise por categoria
3. Integrar clusters do MVP 2 como dimensão na tabela fato da Gold
4. Construir dashboard no Databricks SQL sobre a camada Gold para consumo contínuo
5. Implementar pipeline incremental com atualização batch diária

---
*MVP 4 — Engenharia de Dados · Marcos Lopes Silva · RA 4052024002499 · PUC-Rio · 2026*